In [ ]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
!pip install gcloud
!gcloud auth application-default login

# Import necessary Python libraries
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling
import re                          # Regular expressions

# Note: The actual imports remain exactly as in the original code

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 13.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602927 sha256=0be712086bb442b33c4d4fbcf61beae3dc0a1bbe99dea8ee7cf39c4ab3ea932d
  Stored in directory: /root/.cache/pip/wheels/2a/62/75/3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=MnLv51uTUib6ZSbOswM8UiEdzET1pq&prompt=consent&token_usage=remote&access_type=offline&code_chal

In [ ]:
df = pd.read_excel("/content/PEP_CARREIRAS_ALL_ATT2026.xlsx", decimal=",", thousands=".")
df

,Agrupamentos de Cargos 2,Agrupamentos de Cargos 1,Grupo-Cargo (sigla),Cargo (com código),Escolaridade do Cargo,Cargo em Extinção (indicador),Subsídio (indicador),Plano/Carreira,Nome do Cargo,Código do Cargo,Órgão (Sigla),Qtd. Cargos Ocupados,Qtd. Cargos Aprovados,Qtd. Cargos Distribuidos,Qtd. Cargos Vagos,Remuneração - Mínimo Valor Inicial\n,Remuneração - Máximo Valor Final
0,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,468971.0,722958.0,583255.0,253987.0,1590.73,33721.23
1,ABIN - Agência Brasileira de Inteligência - (C...,Agente de Inteligência - NI (Carreira de Agent...,ABIN,Agente De Inteligencia (617002),NM,0,S,Plano de Carreiras e Cargos da Agência Brasile...,AGENTE DE INTELIGENCIA,617002,ABIN,51.0,937.0,125.0,886.0,6869.43,11805.13
2,ABIN - Agência Brasileira de Inteligência - (C...,Agente Técnico de Inteligência - NI (Carreira ...,ABIN,Agente Tecnico De Inteligencia (617004),NM,0,S,Plano de Carreiras e Cargos da Agência Brasile...,AGENTE TECNICO DE INTELIGENCIA,617004,ABIN,14.0,229.0,47.0,215.0,6181.8,10623.45
3,ABIN - Agência Brasileira de Inteligência - (C...,Oficial de Inteligência - NS (Carreira de Ofic...,ABIN,Oficial De Inteligencia (617001),NS,0,S,Plano de Carreiras e Cargos da Agência Brasile...,OFICIAL DE INTELIGENCIA,617001,ABIN,727.0,2131.0,831.0,1404.0,18116.3,25718.98
4,ABIN - Agência Brasileira de Inteligência - (C...,Oficial Técnico de Inteligência - NS (Carreira...,ABIN,Oficial Tecnico De Inteligencia (617003),NS,0,S,Plano de Carreiras e Cargos da Agência Brasile...,OFICIAL TECNICO DE INTELIGENCIA,617003,ABIN,90.0,439.0,162.0,349.0,16690.89,23144.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13882,-,-,-,-,-,0,-,-,-,400001,MCTI,0.0,1.0,1.0,1.0,-,-
13883,-,-,-,-,-,0,-,-,-,700001,SIPEC,0.0,2.0,0.0,2.0,-,-
13884,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13885,Selection Status:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
rename_map = {
    'Agrupamentos de Cargos 2': 'agrupamentos_de_cargos_2',
    'Agrupamentos de Cargos 1': 'agrupamentos_de_cargos_1',
    'Grupo-Cargo (sigla)': 'grupo-cargo_(sigla)',
    'Cargo (com código)': 'cargo_(com_código)',
    'Escolaridade do Cargo': 'escolaridade_do_cargo',
    'Cargo em Extinção (indicador)': 'cargo_em_extinção_(indicador)',
    'Subsídio (indicador)': 'subsídio_(indicador)',
    'Plano/Carreira': 'plano/carreira',
    'Nome do Cargo': 'nome_do_cargo',
    'Código do Cargo': 'código_do_cargo',
    'Órgão (Sigla)': 'órgão_(sigla)',
    'Qtd. Cargos Ocupados': 'qtd._cargos_ocupados',
    'Qtd. Cargos Aprovados': 'qtd._cargos_aprovados',
    'Qtd. Cargos Distribuidos': 'qtd._cargos_distribuidos',
    'Qtd. Cargos Vagos': 'qtd._cargos_vagos',
    'Remuneração - Mínimo Valor Inicial\n': 'remuneração_-_mínimo_valor_inicial\n',
    'Remuneração - Máximo Valor Final': 'remuneração_-_máximo_valor_final',
}

pep = df.rename(columns=rename_map)

In [ ]:
pep

,agrupamentos_de_cargos_2,agrupamentos_de_cargos_1,grupo-cargo_(sigla),cargo_(com_código),escolaridade_do_cargo,cargo_em_extinção_(indicador),subsídio_(indicador),plano/carreira,nome_do_cargo,código_do_cargo,órgão_(sigla),qtd._cargos_ocupados,qtd._cargos_aprovados,qtd._cargos_distribuidos,qtd._cargos_vagos,remuneração_-_mínimo_valor_inicial\n,remuneração_-_máximo_valor_final
0,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,Total da Seleção,468971.0,722958.0,583255.0,253987.0,1590.73,33721.23
1,ABIN - Agência Brasileira de Inteligência - (C...,Agente de Inteligência - NI (Carreira de Agent...,ABIN,Agente De Inteligencia (617002),NM,0,S,Plano de Carreiras e Cargos da Agência Brasile...,AGENTE DE INTELIGENCIA,617002,ABIN,51.0,937.0,125.0,886.0,6869.43,11805.13
2,ABIN - Agência Brasileira de Inteligência - (C...,Agente Técnico de Inteligência - NI (Carreira ...,ABIN,Agente Tecnico De Inteligencia (617004),NM,0,S,Plano de Carreiras e Cargos da Agência Brasile...,AGENTE TECNICO DE INTELIGENCIA,617004,ABIN,14.0,229.0,47.0,215.0,6181.8,10623.45
3,ABIN - Agência Brasileira de Inteligência - (C...,Oficial de Inteligência - NS (Carreira de Ofic...,ABIN,Oficial De Inteligencia (617001),NS,0,S,Plano de Carreiras e Cargos da Agência Brasile...,OFICIAL DE INTELIGENCIA,617001,ABIN,727.0,2131.0,831.0,1404.0,18116.3,25718.98
4,ABIN - Agência Brasileira de Inteligência - (C...,Oficial Técnico de Inteligência - NS (Carreira...,ABIN,Oficial Tecnico De Inteligencia (617003),NS,0,S,Plano de Carreiras e Cargos da Agência Brasile...,OFICIAL TECNICO DE INTELIGENCIA,617003,ABIN,90.0,439.0,162.0,349.0,16690.89,23144.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13882,-,-,-,-,-,0,-,-,-,400001,MCTI,0.0,1.0,1.0,1.0,-,-
13883,-,-,-,-,-,0,-,-,-,700001,SIPEC,0.0,2.0,0.0,2.0,-,-
13884,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13885,Selection Status:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
pep.keys()

Index(['agrupamentos_de_cargos_2', 'agrupamentos_de_cargos_1',
       'grupo-cargo_(sigla)', 'cargo_(com_código)', 'escolaridade_do_cargo',
       'cargo_em_extinção_(indicador)', 'subsídio_(indicador)',
       'plano/carreira', 'nome_do_cargo', 'código_do_cargo', 'órgão_(sigla)',
       'qtd._cargos_ocupados', 'qtd._cargos_aprovados',
       'qtd._cargos_distribuidos', 'qtd._cargos_vagos',
       'remuneração_-_mínimo_valor_inicial\n',
       'remuneração_-_máximo_valor_final'],
      dtype='object')

In [ ]:
# Filter the data
pep = pep.iloc[:-3]
pep = pep[pep['cargo_em_extinção_(indicador)'] == 0]


In [ ]:
pep.dropna()

,agrupamentos_de_cargos_2,agrupamentos_de_cargos_1,grupo-cargo_(sigla),cargo_(com_código),escolaridade_do_cargo,cargo_em_extinção_(indicador),subsídio_(indicador),plano/carreira,nome_do_cargo,código_do_cargo,órgão_(sigla),qtd._cargos_ocupados,qtd._cargos_aprovados,qtd._cargos_distribuidos,qtd._cargos_vagos,remuneração_-_mínimo_valor_inicial\n,remuneração_-_máximo_valor_final
1,ABIN - Agência Brasileira de Inteligência - (C...,Agente de Inteligência - NI (Carreira de Agent...,ABIN,Agente De Inteligencia (617002),NM,0,S,Plano de Carreiras e Cargos da Agência Brasile...,AGENTE DE INTELIGENCIA,617002,ABIN,51.0,937.0,125.0,886.0,6869.43,11805.13
2,ABIN - Agência Brasileira de Inteligência - (C...,Agente Técnico de Inteligência - NI (Carreira ...,ABIN,Agente Tecnico De Inteligencia (617004),NM,0,S,Plano de Carreiras e Cargos da Agência Brasile...,AGENTE TECNICO DE INTELIGENCIA,617004,ABIN,14.0,229.0,47.0,215.0,6181.8,10623.45
3,ABIN - Agência Brasileira de Inteligência - (C...,Oficial de Inteligência - NS (Carreira de Ofic...,ABIN,Oficial De Inteligencia (617001),NS,0,S,Plano de Carreiras e Cargos da Agência Brasile...,OFICIAL DE INTELIGENCIA,617001,ABIN,727.0,2131.0,831.0,1404.0,18116.3,25718.98
4,ABIN - Agência Brasileira de Inteligência - (C...,Oficial Técnico de Inteligência - NS (Carreira...,ABIN,Oficial Tecnico De Inteligencia (617003),NS,0,S,Plano de Carreiras e Cargos da Agência Brasile...,OFICIAL TECNICO DE INTELIGENCIA,617003,ABIN,90.0,439.0,162.0,349.0,16690.89,23144.49
5,ABIN - Agência Brasileira de Inteligência - (G...,Cargos de Nível Auxiliar - NA do Grupo Apoio d...,GAA,Auxiliar Operacional Servicos Diversos (620036),NF,0,N,Plano de Carreiras e Cargos da Agência Brasile...,AUXILIAR OPERACIONAL SERVICOS DIVERSOS,620036,ABIN,1.0,1.0,1.0,0.0,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13879,Tribunal Marítimo,Juiz do Tribunal Marítimo - NS,PCJTM,Juiz Do Tribunal Maritimo (905001),NS,0,N,Sem Plano,JUIZ DO TRIBUNAL MARITIMO,905001,CM,6.0,6.0,6.0,0.0,-,-
13880,Tribunal Marítimo,Juiz-Presidente do Tribunal Marítimo - NS,PCJTM,Juiz-Presidente Do Tribunal Maritimo (905002),NS,0,N,Sem Plano,JUIZ-PRESIDENTE DO TRIBUNAL MARITIMO,905002,CM,1.0,1.0,1.0,0.0,-,-
13881,-,-,-,-,-,0,-,-,-,400004,FIOCRUZ,0.0,1.0,1.0,1.0,-,-
13882,-,-,-,-,-,0,-,-,-,400001,MCTI,0.0,1.0,1.0,1.0,-,-


In [ ]:
indicador = pep.groupby('agrupamentos_de_cargos_1').agg(n=('qtd._cargos_ocupados', 'sum')).reset_index()

In [ ]:
indicador

,agrupamentos_de_cargos_1,n
0,-,0.0
1,Advogado da União - Lotado na AGU - NS (Carrei...,1860.0
2,Agente Executivo da CVM - NI (Carreira de Agen...,89.0
3,Agente Executivo da CVM/SUSEP - NI (Carreira d...,2.0
4,Agente Executivo da SUSEP - NI (Carreira de Ag...,54.0
...,...,...
235,Técnico em Regulação de Serviços Públicos de T...,325.0
236,Técnico em Regulação de Serviços de Transporte...,101.0
237,Técnico em Regulação de Serviços de Transporte...,479.0
238,Técnico em Regulação e Vigilância Sanitária - ...,80.0


In [ ]:
# Definição das faixas de servidores
bins = [0, 99, 499, 1000, 5000, np.inf]
labels = ['Até 99', '100 a 499', '500 a 1.000', '1.001 a 5.000', 'Acima de 5 mil']
indicador['faixa_servidores'] = pd.cut(indicador['n'], bins=bins, labels=labels, right=False)
indicador

,agrupamentos_de_cargos_1,n,faixa_servidores
0,-,0.0,Até 99
1,Advogado da União - Lotado na AGU - NS (Carrei...,1860.0,1.001 a 5.000
2,Agente Executivo da CVM - NI (Carreira de Agen...,89.0,Até 99
3,Agente Executivo da CVM/SUSEP - NI (Carreira d...,2.0,Até 99
4,Agente Executivo da SUSEP - NI (Carreira de Ag...,54.0,Até 99
...,...,...,...
235,Técnico em Regulação de Serviços Públicos de T...,325.0,100 a 499
236,Técnico em Regulação de Serviços de Transporte...,101.0,100 a 499
237,Técnico em Regulação de Serviços de Transporte...,479.0,100 a 499
238,Técnico em Regulação e Vigilância Sanitária - ...,80.0,Até 99


In [ ]:
indicador.rename(columns={'agrupamentos_de_cargos_1': 'agrupamentos'}, inplace=True)

In [ ]:
indicador = indicador.rename(columns={'n': 'quantidade_servidores'})

In [ ]:
indicador['quantidade_servidores'] = 1
indicador

,agrupamentos,quantidade_servidores,faixa_servidores
0,-,1,Até 99
1,Advogado da União - Lotado na AGU - NS (Carrei...,1,1.001 a 5.000
2,Agente Executivo da CVM - NI (Carreira de Agen...,1,Até 99
3,Agente Executivo da CVM/SUSEP - NI (Carreira d...,1,Até 99
4,Agente Executivo da SUSEP - NI (Carreira de Ag...,1,Até 99
...,...,...,...
235,Técnico em Regulação de Serviços Públicos de T...,1,100 a 499
236,Técnico em Regulação de Serviços de Transporte...,1,100 a 499
237,Técnico em Regulação de Serviços de Transporte...,1,100 a 499
238,Técnico em Regulação e Vigilância Sanitária - ...,1,Até 99


In [ ]:
indicador

,agrupamentos,quantidade_servidores,faixa_servidores
0,-,1,Até 99
1,Advogado da União - Lotado na AGU - NS (Carrei...,1,1.001 a 5.000
2,Agente Executivo da CVM - NI (Carreira de Agen...,1,Até 99
3,Agente Executivo da CVM/SUSEP - NI (Carreira d...,1,Até 99
4,Agente Executivo da SUSEP - NI (Carreira de Ag...,1,Até 99
...,...,...,...
235,Técnico em Regulação de Serviços Públicos de T...,1,100 a 499
236,Técnico em Regulação de Serviços de Transporte...,1,100 a 499
237,Técnico em Regulação de Serviços de Transporte...,1,100 a 499
238,Técnico em Regulação e Vigilância Sanitária - ...,1,Até 99


In [ ]:
# Contagem e cálculo de porcentagens por faixa de servidores
indicador1 = indicador.groupby('faixa_servidores').size().reset_index(name='qtde_planos')
indicador1['perc_planos'] = 100 * indicador1['qtde_planos'] / indicador1['qtde_planos'].sum()
indicador1

/tmp/ipykernel_959/1632690437.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  indicador1 = indicador.groupby('faixa_servidores').size().reset_index(name='qtde_planos')


,faixa_servidores,qtde_planos,perc_planos
0,Até 99,120,50.000000
1,100 a 499,62,25.833333
2,500 a 1.000,22,9.166667
3,1.001 a 5.000,25,10.416667
4,Acima de 5 mil,11,4.583333


In [ ]:
indicador.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   agrupamentos           240 non-null    object  
 1   quantidade_servidores  240 non-null    int64   
 2   faixa_servidores       240 non-null    category
dtypes: category(1), int64(1), object(1)
memory usage: 4.3+ KB


In [ ]:
df = indicador1

In [ ]:
df['ano'] = 2026

# Consumindo a base anterior para agregar o novo ano

In [ ]:
query = """SELECT * FROM `repositoriodedadosgpsp.estrutura_organizacional_carreiras.PEP_faixa_servidores`"""
# Execute the query using pandas_gbq.read_gbq and load the result into a pandas DataFrame called 'df'.
# The 'project_id' specifies the Google Cloud Project to use.
df_old = pandas_gbq.read_gbq(query, project_id='repositoriodedadosgpsp')

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Downloading: 100%|██████████|


In [ ]:
df_old['ano'] = 2023

In [ ]:
df_old.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   faixa_servidores  5 non-null      object 
 1   qtde_planos       5 non-null      Int64  
 2   perc_planos       5 non-null      float64
 3   ano               5 non-null      int64  
dtypes: Int64(1), float64(1), int64(1), object(1)
memory usage: 297.0+ bytes


In [ ]:
df_final = pd.concat([df, df_old], ignore_index=True)

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   faixa_servidores  10 non-null     object 
 1   qtde_planos       10 non-null     Int64  
 2   perc_planos       10 non-null     float64
 3   ano               10 non-null     int64  
dtypes: Int64(1), float64(1), int64(1), object(1)
memory usage: 462.0+ bytes


# Upando informações para o GBQ

In [ ]:
# Define the BigQuery table schema with Portuguese descriptions
schema = [
    bigquery.SchemaField('ano','INTEGER',description='Ano de referência da observação'),
    bigquery.SchemaField('perc_planos', 'FLOAT', description='Percentual de servidores por plano'),
    bigquery.SchemaField('qtde_planos', 'INTEGER', description='Quantidade de servidores por plano'),
    bigquery.SchemaField('faixa_servidores', 'STRING', description='Faixa de servidores por plano')
]
# Initialize BigQuery client connection
client = bigquery.Client(project='repositoriodedadosgpsp')

# Create reference to target dataset
dataset_ref = client.dataset('estrutura_organizacional_carreiras')
# Create reference to target table with standardized naming convention:
# FONTE_algo_intuitivo_dado (MUNIC_quantidade_vinculos_mapa_v1)
table_ref = dataset_ref.table('PEP_faixa_servidores_v1')

# Configure the load job with our schema definition
job_config = bigquery.LoadJobConfig(
    schema=schema,
    # Optional parameters (commented out):
    # write_disposition="WRITE_TRUNCATE",  # Overwrites table if exists
    # create_disposition="CREATE_IF_NEEDED"  # Default behavior
)

# Execute the load job to upload DataFrame to BigQuery
job = client.load_table_from_dataframe(
    dataframe=df_final,
    destination=table_ref,
    job_config=job_config
)

# Wait for the job to complete
job.result()

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


LoadJob<project=repositoriodedadosgpsp, location=US, id=d7ccfdc2-3d06-4b2d-8529-2e89afc01a6c>